# 🎨 MiniMax-H3-Image 开源本地推理 Colab 实战（免 ComfyUI · 极速直拉版）

本 Notebook 演示如何在 **Google Colab (L4 24GB / A100)** 上以 **100% 开源本地推理** 运行 **MiniMax-H3-Image**（33.1B DiT + Qwen3-VL-32B + 2D ViT VAE，6 步 PDD 蒸馏，原生最高 4K / 16MP），**无需启动 ComfyUI**：

### ⚡ 为什么直接拉取 > 谷歌云盘加载？（Google Drive 最佳实践）
- **网络极速直拉**：Colab 实例与 Hugging Face 之间走 Google Cloud 数据中心高速骨干网，直拉下行速度通常达 **100 ~ 300+ MB/s**，下载 ~21GB 主干模型通常仅需 **1~2 分钟**。
- **谷歌网盘 FUSE 瓶颈**：通过 `/content/drive/MyDrive` 挂载读写几十 GB 大文件会遭遇 Google Drive API 限速与 FUSE 转换损耗（通常仅 20~30 MB/s），从网盘拷贝大文件往往比直接联网拉取**慢数倍**，且容易出现 I/O 挂起。
- **清晰的职责分工**：
  1. **模型权重**：直接高速拉取到本地 NVMe 临时存储（`/content/models`），不占用高延迟的网盘 FUSE 管道；
  2. **二进制编译缓存**：`sd-cli` 可执行文件仅 ~20MB，从网盘缓存恢复只需 **0.5 秒**，免除每次开机 3~5 分钟的 CUDA 编译；
  3. **成果持久化**：生成的高清图像直接输出到 Google Drive（`/content/drive/MyDrive/MiniMax_Outputs`），实例断开也不丢图；
  4. **大文件可选备份**：提供 `BACKUP_WEIGHTS_TO_DRIVE = False` 开关，仅在用户有明确冷备需求时可选开启。

In [ ]:
# ==========================================
# Cell 1: 全局配置与 2 小时超时防扣费看门狗 (Watchdog)
# ==========================================
import os
import threading
import time
from pathlib import Path

# Google Drive 持久化目录（用于保存最终成图与 ~20MB 编译二进制）
DRIVE_OUTPUTS_DIR = Path("/content/drive/MyDrive/MiniMax_Outputs/H3_Image")
DRIVE_BIN_DIR = Path("/content/drive/MyDrive/MiniMax_Models/bin")
DRIVE_MODELS_BACKUP_DIR = Path("/content/drive/MyDrive/MiniMax_Models/H3_Image")

# 本地高速 NVMe 模型运行目录（避免网盘 FUSE 慢速与 I/O 锁）
LOCAL_MODELS_DIR = Path("/content/models")

# 是否将 ~40GB 大模型权重备份到 Google Drive（默认 False：直连下载远快于网盘读写）
BACKUP_WEIGHTS_TO_DRIVE = False
# 跑完演示是否自动断开 Colab 释放 GPU 算力单元 (Compute Units)
AUTO_SHUTDOWN = False
MAX_RUNTIME_SECONDS = 7200  # 2 小时硬性超时看门狗

def safe_shutdown(reason: str = "Task finished"):
    print(f"\n🔌 [Auto-Shutdown] {reason}，正在释放 Colab GPU 算力单元...")
    time.sleep(3)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"⚠️ 无法调用 runtime.unassign(): {e}，执行进程终止...")
        os.kill(os.getpid(), 9)

def _watchdog():
    time.sleep(MAX_RUNTIME_SECONDS)
    safe_shutdown(f"达到最大运行时长 ({MAX_RUNTIME_SECONDS}s) 保护上限")

threading.Thread(target=_watchdog, daemon=True).start()
print("✅ 全局配置已就绪，2 小时超时看门狗已在后台启动。")

In [ ]:
# ==========================================
# Cell 2: 挂载 Google Drive 与初始化目录结构
# ==========================================
from google.colab import drive

drive.mount("/content/drive")

# 创建网盘输出目录与本地 NVMe 目录
DRIVE_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_BIN_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["diffusion_models", "vae", "text_encoders"]:
    (LOCAL_MODELS_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"✅ 生成图片保存路径 (Google Drive) : {DRIVE_OUTPUTS_DIR}")
print(f"✅ 编译二进制缓存路径 (Google Drive) : {DRIVE_BIN_DIR}")
print(f"✅ 本地 NVMe 高速模型目录          : {LOCAL_MODELS_DIR}")

In [ ]:
# ==========================================
# Cell 3: GPU 算力门禁检测 (确保为 L4 24GB 或 A100 40GB/80GB)
# ==========================================
import subprocess

def check_colab_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()[0]
    except Exception as e:
        raise RuntimeError(
            "❌ 未检测到 NVIDIA GPU！请在 Colab 顶部菜单选择：\n"
            "『代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type) -> 选择 L4 GPU 或 A100 GPU』"
        ) from e

    name, cc_str, mem_str = [x.strip() for x in out.split(",")]
    cc = float(cc_str)
    vram_gb = float(mem_str) / 1024.0
    print(f"🔍 检测到 GPU: {name} | Compute Capability: {cc} | VRAM: {vram_gb:.1f} GB")
    if cc < 8.0 or vram_gb < 20.0:
        raise RuntimeError(
            f"❌ 当前 GPU ({name}, {vram_gb:.1f}GB, CC {cc}) 不满足 MiniMax-H3-Image 最低要求。\n"
            "请切换至 Colab L4 (24GB) 或 A100 (40GB/80GB)。"
        )
    return name, cc_str.replace(".", ""), vram_gb

GPU_NAME, GPU_SM, GPU_VRAM_GB = check_colab_gpu()

In [ ]:
# ==========================================
# Cell 4: 编译或从 Google Drive 秒级恢复 sd-cli (仅 ~20MB 二进制，极速同步)
# ==========================================
import os
import shutil
import subprocess
import sys

SDCPP_REPO_DIR = Path("/content/stable-diffusion.cpp")
SD_CLI_LOCAL = SDCPP_REPO_DIR / "build/bin/sd-cli"
SD_CLI_CACHED = DRIVE_BIN_DIR / f"sd-cli-h3image-sm{GPU_SM}"

def run_streaming(cmd: list[str]) -> None:
    """实时流式输出子进程日志（如 Ninja 编译进度条 [x/y]），绝不卡顿盲等。"""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

if SD_CLI_CACHED.exists() and SD_CLI_CACHED.stat().st_size > 1024 * 1024:
    SD_CLI_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SD_CLI_CACHED, SD_CLI_LOCAL)
    SD_CLI_LOCAL.chmod(0o755)
    print(f"⚡ 已从 Google Drive 缓存秒级恢复 sd-cli (仅耗时 <1 秒): {SD_CLI_CACHED}")
else:
    print(f"🔨 首次在 {GPU_NAME} (sm_{GPU_SM}) 上运行，正在拉取并编译 stable-diffusion.cpp (约 3~5 分钟)...")
    run_streaming(["apt-get", "update", "-qq"])
    run_streaming(["apt-get", "install", "-y", "-qq", "cmake", "ninja-build"])
    if not SDCPP_REPO_DIR.exists():
        run_streaming([
            "git", "clone", "--depth", "1", "--recursive",
            "https://github.com/leejet/stable-diffusion.cpp", str(SDCPP_REPO_DIR)
        ])
    run_streaming([
        "cmake", "-S", str(SDCPP_REPO_DIR), "-B", str(SDCPP_REPO_DIR / "build"),
        "-GNinja",
        "-DSD_CUDA=ON",
        f"-DCMAKE_CUDA_ARCHITECTURES={GPU_SM}",
        "-DCMAKE_BUILD_TYPE=Release",
    ])
    run_streaming([
        "cmake", "--build", str(SDCPP_REPO_DIR / "build"), "--target", "sd-cli", "-j", str(os.cpu_count() or 4)
    ])
    shutil.copy2(SD_CLI_LOCAL, SD_CLI_CACHED)
    print(f"✅ 编译产物已成功缓存至 Google Drive: {SD_CLI_CACHED}")

run_streaming([str(SD_CLI_LOCAL), "--version"])

In [ ]:
# ==========================================
# Cell 5: 直连 Hugging Face 高速拉取权重到本地 NVMe (带原生 tqdm 进度条)
# ==========================================
# 💡 关键机制：显式关闭 hf_transfer，恢复 Hugging Face 原生 tqdm 进度条（实时显示下载速度 MB/s 与剩余时间）
!pip install -q -U huggingface_hub tqdm Pillow
import os
import shutil
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm.auto import tqdm

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def copy_with_progress(src: Path, dst: Path, desc: str) -> None:
    """带进度条的大文件可选备份。"""
    dst.parent.mkdir(parents=True, exist_ok=True)
    total_bytes = src.stat().st_size
    chunk_size = 32 * 1024 * 1024  # 32MB 分块
    with open(src, "rb") as fsrc, open(dst, "wb") as fdst, tqdm(
        total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024, desc=desc
    ) as pbar:
        while True:
            buf = fsrc.read(chunk_size)
            if not buf:
                break
            fdst.write(buf)
            pbar.update(len(buf))

def ensure_weight_fast(repo_id: str, filename: str, local_subdir: str) -> Path:
    """
    高速拉取核心逻辑：
    1. 优先使用本地高速 NVMe 现有文件（0 秒就绪）；
    2. 否则直接通过 Google 数据中心内网高带宽从 Hugging Face 极速拉取（最快路径）；
    3. 仅在 BACKUP_WEIGHTS_TO_DRIVE=True 时才可选做 Drive 备份，绝不阻塞默认启动流程。
    """
    target_name = Path(filename).name
    local_path = LOCAL_MODELS_DIR / local_subdir / target_name
    if local_path.exists() and local_path.stat().st_size > 1024 * 1024:
        size_gb = local_path.stat().st_size / (1024**3)
        print(f"✅ [本地 NVMe 已就绪] {target_name} ({size_gb:.2f} GB)")
        return local_path

    print(f"⬇️ 正在从 Hugging Face ({repo_id}) 高速拉取到本地 NVMe: {filename} ...")
    downloaded = Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=str(LOCAL_MODELS_DIR / "_hf_cache"),
        )
    )
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(downloaded), str(local_path))
    size_gb = local_path.stat().st_size / (1024**3)
    print(f"✅ 拉取并校验完成: {target_name} ({size_gb:.2f} GB)")

    if BACKUP_WEIGHTS_TO_DRIVE:
        drive_backup = DRIVE_MODELS_BACKUP_DIR / local_subdir / target_name
        copy_with_progress(local_path, drive_backup, desc=f"💾 可选备份至 Google Drive: {target_name}")
    return local_path

# 1. DiT 主干 (INT8 + Hadamard ConvRot, ~21 GB，直拉约 1~2 分钟)
DIT_PATH = ensure_weight_fast(
    "Comfy-Org/MiniMax-H3-Image",
    "diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors",
    "diffusion_models",
)

# 2. 2D Image VAE (FP16, ~1.17 GB，直拉约 5 秒)
VAE_PATH = ensure_weight_fast(
    "Comfy-Org/MiniMax-H3-Image",
    "vae/minimax_h3_image_vae_fp16.safetensors",
    "vae",
)

# 3. Qwen3-VL-32B 文本编码器 (Q4_K_M GGUF，直拉约 1 分钟)
LLM_VISION_PATH = None
try:
    gguf_files = list_repo_files("leejet/MiniMax-H3-GGUF")
    q4_candidates = [f for f in gguf_files if "qwen3vl" in f.lower() and "q4_k_m" in f.lower() and f.endswith(".gguf")]
    mmproj_candidates = [f for f in gguf_files if ("mmproj" in f.lower() or "vision" in f.lower()) and f.endswith(".gguf")]
    llm_filename = q4_candidates[0] if q4_candidates else "qwen3vl_32b_minimax_h3-Q4_K_M.gguf"
    LLM_PATH = ensure_weight_fast("leejet/MiniMax-H3-GGUF", llm_filename, "text_encoders")
    if mmproj_candidates:
        LLM_VISION_PATH = ensure_weight_fast("leejet/MiniMax-H3-GGUF", mmproj_candidates[0], "text_encoders")
except Exception as e:
    print(f"ℹ️ GGUF 探测回退至 Comfy-Org INT8 safetensors: {e}")
    LLM_PATH = ensure_weight_fast(
        "Comfy-Org/MiniMax-H3",
        "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
        "text_encoders",
    )

print("\n🎉 所有模型权重已全部高速就绪于本地 NVMe：")
print(f"  - DiT  : {DIT_PATH}")
print(f"  - VAE  : {VAE_PATH}")
print(f"  - LLM  : {LLM_PATH}")
if LLM_VISION_PATH:
    print(f"  - Vision: {LLM_VISION_PATH}")

In [ ]:
# ==========================================
# Cell 6: 封装 sd-cli + Demo 1 文生图 (Text-to-Image)
# ==========================================
import time
from IPython.display import Image as IPyImage, display

def run_minimax_h3_image_sdcpp(
    prompt: str,
    output_path: Path,
    width: int = 2048,
    height: int = 2048,
    seed: int = 42,
    ref_images: list[Path] | None = None,
) -> Path:
    if width % 32 != 0 or height % 32 != 0:
        raise ValueError(f"宽高必须是 32 的整数倍，当前指定为 {width}x{height}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(SD_CLI_LOCAL),
        "--diffusion-model", str(DIT_PATH),
        "--vae", str(VAE_PATH),
        "--llm", str(LLM_PATH),
    ]
    if LLM_VISION_PATH:
        cmd.extend(["--llm_vision", str(LLM_VISION_PATH)])
    for ref in (ref_images or []):
        cmd.extend(["-r", str(ref)])
    cmd.extend([
        "-p", prompt,
        "--cfg-scale", "1.0",
        "--steps", "6",
        "--sampling-method", "euler",
        "--sigmas", "minimax-h3-image",
        "--seed", str(seed),
        "-W", str(width),
        "-H", str(height),
        "--offload-to-cpu",
        "--rng", "cpu",
        "--flash-attn", "off,diffusion=on",
        "--vae-tiling",
        "-v",
        "-o", str(output_path),
    ])
    print("🚀 执行 sd-cli 推理命令:\n", " ".join(cmd))
    t0 = time.time()
    run_streaming(cmd)
    print(f"✅ 生成成功 ({width}x{height})，耗时 {time.time() - t0:.1f}s -> 保存至网盘: {output_path}")
    return output_path

T2I_PROMPT = (
    "Editorial architectural photograph of a cantilevered glass tea pavilion suspended above a misty "
    "alpine canyon at blue hour. Warm amber lantern light glows through minimal cedar latticework, "
    "reflecting on a still infinity water basin. Crisp wet slate textures in the foreground, drifted "
    "mountain fog in the midground, and snow-dusted granite peaks under a twilight indigo sky. "
    "Shot on medium-format digital camera, 45mm tilt-shift lens, f/8, razor-sharp micro-detail."
)

# 结果直接写入 Google Drive 目录，断开连接也不会丢失
t2i_out = DRIVE_OUTPUTS_DIR / "demo1_t2i_2048.png"
run_minimax_h3_image_sdcpp(
    prompt=T2I_PROMPT,
    output_path=t2i_out,
    width=2048,
    height=2048,
    seed=42,
)
display(IPyImage(filename=str(t2i_out), width=640))

In [ ]:
# ==========================================
# Cell 7: Demo 2 多图参考/图像编辑 (<Picture 1> 语法)
# ==========================================
# 直接将 Demo 1 生成的网盘图片作为 <Picture 1> 参考图进行季节与光影编辑
EDIT_PROMPT = (
    "Transform the architectural scene in <Picture 1> into peak autumn at golden sunrise: "
    "surround the glass tea pavilion with vibrant crimson maple foliage, warm golden sunbeams "
    "piercing through the canyon mist, preserving the exact pavilion geometry and water reflection."
)

edit_out = DRIVE_OUTPUTS_DIR / "demo2_edit_2048.png"
run_minimax_h3_image_sdcpp(
    prompt=EDIT_PROMPT,
    output_path=edit_out,
    width=2048,
    height=2048,
    seed=42,
    ref_images=[t2i_out],
)
display(IPyImage(filename=str(edit_out), width=640))

In [ ]:
# ==========================================
# Cell 8 (可选): 纯 Python Hugging Face Diffusers ModularPipeline (需 A100 80GB)
# ==========================================
RUN_DIFFUSERS_DEMO = False  # 若当前是 Colab A100 80GB 且想体验纯 Python 管线，改为 True

if RUN_DIFFUSERS_DEMO:
    if GPU_VRAM_GB < 60:
        print(f"⚠️ 当前显存为 {GPU_VRAM_GB:.1f} GB，Diffusers BF16 原生路线峰值显存约 65 GB，请使用上方 sd-cli 路线！")
    else:
        run_streaming([
            "pip", "install", "-q", "-U",
            "git+https://github.com/huggingface/diffusers.git", "transformers>=5.0.0", "accelerate", "safetensors"
        ])
        import torch
        from diffusers import ComponentsManager, ModularPipeline

        manager = ComponentsManager()
        manager.enable_auto_cpu_offload(device="cuda")
        pipe = ModularPipeline.from_pretrained("MiniMaxAI/MiniMax-H3-Image", components_manager=manager)
        pipe.load_components(dtype=torch.bfloat16)

        image = pipe(
            prompt=T2I_PROMPT,
            height=4096,
            width=4096,
            generator=torch.Generator(device="cpu").manual_seed(42),
            output="images",
        )[0]
        diff_out = DRIVE_OUTPUTS_DIR / "demo3_diffusers_4096.png"
        image.save(diff_out)
        print(f"✅ Diffusers 4096x4096 生成完成并保存至网盘: {diff_out}")
        display(IPyImage(filename=str(diff_out), width=640))

if AUTO_SHUTDOWN:
    safe_shutdown("所有演示任务已完成")